# 00 — Preprocessing & Binning Target (Age → Age_Group)

**Dataset**: Biological Age Predictive (Kaggle)  
**Tujuan**: Menyiapkan data yang sudah di-preprocessing dan melakukan **binning** pada target `Age (years)` agar masalah regresi berubah menjadi **klasifikasi**.

### Alur Notebook
1. Load data Train & Test
2. Ulangi semua preprocessing yang sudah dilakukan sebelumnya
3. Analisis distribusi umur
4. Lakukan binning → `Age_Group`
5. Tampilkan distribusi kelas akhir
6. Simpan hasil untuk notebook skenario berikutnya

In [ ]:
# ============================================================
# CELL 1: Import library yang dibutuhkan
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

# Atur style visualisasi
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Library berhasil di-import ✅')

In [ ]:
# ============================================================
# CELL 2: Load dataset Train dan Test
# ============================================================
# Sesuaikan path sesuai lokasi dataset
DATA_DIR = 'Dataset'

df_train = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))

# CATATAN: Test.csv dari Kaggle TIDAK memiliki kolom target 'Age (years)'
# karena file ini digunakan untuk submission prediksi (hanya berisi fitur).
# Oleh karena itu, Test.csv memiliki 26 kolom sedangkan Train.csv memiliki 27.
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'Test.csv'))

print(f'Train shape: {df_train.shape}')
print(f'Test shape : {df_test.shape}')
print(f'\nKolom: {list(df_train.columns)}')

---
## Bagian A — Preprocessing (sama seperti notebook sebelumnya)
Langkah-langkah berikut **mengulang** preprocessing yang sudah dilakukan agar data konsisten.

In [ ]:
# ============================================================
# CELL 3: F.1 — Hapus kolom ID
# ============================================================
df_train = df_train.drop(columns=['ID'])
df_test  = df_test.drop(columns=['ID'])
print('Kolom ID berhasil dihapus ✅')
print(f'Train shape: {df_train.shape} | Test shape: {df_test.shape}')

In [ ]:
# ============================================================
# CELL 4: F.2 — Parsing Blood Pressure (s/d) → 2 kolom numerik
# ============================================================
def parse_blood_pressure(df):
    bp_split = df['Blood Pressure (s/d)'].str.split('/', expand=True).astype(float)
    df['BP_Systolic']  = bp_split[0]
    df['BP_Diastolic'] = bp_split[1]
    df = df.drop(columns=['Blood Pressure (s/d)'])
    return df

df_train = parse_blood_pressure(df_train)
df_test  = parse_blood_pressure(df_test)
print('Blood Pressure berhasil diparsing ✅')

In [ ]:
# ============================================================
# CELL 7: F.5 — Encoding variabel kategorikal
# ============================================================

# --- Label Encoding untuk kolom ordinal/binary ---
label_encode_cols = {
    'Gender'                : ['Female', 'Male'],
    'Physical Activity Level': ['Low', 'Moderate', 'High'],
    'Smoking Status'         : ['Never', 'Former', 'Current'],
    'Alcohol Consumption'    : ['Unknown', 'Occasional', 'Frequent'],
    'Medication Use'         : ['Unknown', 'Occasional', 'Regular'],
    'Mental Health Status'   : ['Poor', 'Fair', 'Good', 'Excellent'],
    'Sleep Patterns'         : ['Insomnia', 'Normal', 'Excessive'],
    'Income Level'           : ['Low', 'Medium', 'High'],
    'Education Level'        : ['Unknown', 'High School', 'Undergraduate', 'Postgraduate'],
}

for col, order in label_encode_cols.items():
    mapping = {val: i for i, val in enumerate(order)}
    df_train[col] = df_train[col].map(mapping)
    df_test[col]  = df_test[col].map(mapping)

# --- Validasi: cek apakah ada NaN baru setelah Label Encoding ---
# NaN bisa muncul jika ada nilai di data yang tidak tercantum di mapping
nan_train = df_train[list(label_encode_cols.keys())].isnull().sum()
nan_test  = df_test[list(label_encode_cols.keys())].isnull().sum()
nan_issues = nan_train[nan_train > 0].add(nan_test[nan_test > 0], fill_value=0)
if len(nan_issues) > 0:
    print('⚠️ PERINGATAN: Ditemukan NaN baru setelah Label Encoding!')
    print('Kolom bermasalah:')
    for col_name, cnt in nan_issues.items():
        print(f'  - {col_name}: {int(cnt)} NaN')
else:
    print('Validasi Label Encoding: tidak ada NaN baru ✅')

# --- One-Hot Encoding untuk kolom nominal ---
ohe_cols = ['Diet', 'Chronic Diseases', 'Family History']

df_train = pd.get_dummies(df_train, columns=ohe_cols, drop_first=False)
df_test  = pd.get_dummies(df_test,  columns=ohe_cols, drop_first=False)

# Selaraskan kolom train dan test
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

print(f'Shape setelah encoding — Train: {df_train.shape} | Test: {df_test.shape}')
print('Encoding selesai ✅')

In [ ]:
# ============================================================
# CELL 6: F.4 — Imputasi missing values kategorikal → 'Unknown'
# ============================================================
cols_with_missing = ['Alcohol Consumption', 'Chronic Diseases', 'Medication Use',
                     'Family History', 'Education Level']

for col in cols_with_missing:
    df_train[col] = df_train[col].fillna('Unknown')
    df_test[col]  = df_test[col].fillna('Unknown')

print(f'Missing values tersisa (train): {df_train.isnull().sum().sum()}')
print(f'Missing values tersisa (test) : {df_test.isnull().sum().sum()}')
print('Imputasi selesai ✅')

In [ ]:
# ============================================================
# CELL 7: F.5 — Encoding variabel kategorikal
# ============================================================

# --- Label Encoding untuk kolom ordinal/binary ---
label_encode_cols = {
    'Gender'                : ['Female', 'Male'],
    'Physical Activity Level': ['Low', 'Moderate', 'High'],
    'Smoking Status'         : ['Never', 'Former', 'Current'],
    'Alcohol Consumption'    : ['Unknown', 'Occasional', 'Frequent'],
    'Medication Use'         : ['Unknown', 'Occasional', 'Regular'],
    'Mental Health Status'   : ['Poor', 'Fair', 'Good', 'Excellent'],
    'Sleep Patterns'         : ['Insomnia', 'Normal', 'Excessive'],
    'Income Level'           : ['Low', 'Medium', 'High'],
    'Education Level'        : ['Unknown', 'High School', 'Undergraduate', 'Postgraduate'],
}

for col, order in label_encode_cols.items():
    mapping = {val: i for i, val in enumerate(order)}
    df_train[col] = df_train[col].map(mapping)
    df_test[col]  = df_test[col].map(mapping)

# --- One-Hot Encoding untuk kolom nominal ---
ohe_cols = ['Diet', 'Chronic Diseases', 'Family History']

df_train = pd.get_dummies(df_train, columns=ohe_cols, drop_first=False)
df_test  = pd.get_dummies(df_test,  columns=ohe_cols, drop_first=False)

# Selaraskan kolom train dan test
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

print(f'Shape setelah encoding — Train: {df_train.shape} | Test: {df_test.shape}')
print('Encoding selesai ✅')

### Verifikasi Kuartil
Cell berikut menampilkan nilai kuartil aktual dari data training untuk memvalidasi kesesuaian batas bin yang telah dipilih di atas.

In [ ]:
# ============================================================
# Validasi kuartil aktual vs batas bin yang dipilih
# ============================================================
print('=== Kuartil Aktual y_train_raw ===')
print(y_train_raw.quantile([0.25, 0.5, 0.75]))

# Bandingkan dengan batas bin: [17, 35, 53, 71, 90]
# Q1 ≈ 36 → batas bin ke-1 = 35 (selisih ~1 tahun)
# Q2 ≈ 53 → batas bin ke-2 = 53 (tepat sama)
# Q3 ≈ 72 → batas bin ke-3 = 71 (selisih ~1 tahun)
# Kesimpulan: batas bin yang dipilih sangat selaras dengan kuartil data,
# sehingga distribusi kelas terjamin relatif seimbang.
print('\nBatas bin dipilih : [17, 35, 53, 71, 90]')
print('Kuartil data      : Q1≈36, Q2≈53, Q3≈72')
print('→ Batas bin selaras dengan kuartil data ✅')

In [ ]:
# ============================================================
# CELL 8: F.6 — Feature Scaling (StandardScaler)
# ============================================================
TARGET = 'Age (years)'

# Pisahkan fitur dan target
X_train = df_train.drop(columns=[TARGET])
y_train_raw = df_train[TARGET].copy()  # simpan target asli sebelum binning
X_test  = df_test.drop(columns=[TARGET], errors='ignore')

# Konversi kolom boolean ke int
bool_cols = X_train.select_dtypes(include='bool').columns.tolist()
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols]  = X_test[bool_cols].astype(int)

# Scaling kolom numerik
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f'X_train shape : {X_train.shape}')
print(f'y_train shape : {y_train_raw.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'Target range  : {y_train_raw.min()} – {y_train_raw.max()} tahun')
print('\nPreprocessing selesai ✅')

---
## Bagian B — Analisis Distribusi Umur
Sebelum menentukan strategi binning, kita perlu memahami distribusi umur pada data training.

In [ ]:
# ============================================================
# CELL 9: Statistik deskriptif target Age (years)
# ============================================================
print('=== Statistik Deskriptif Age (years) ===')
print(y_train_raw.describe())
print(f'\nMedian   : {y_train_raw.median()}')
print(f'Skewness : {y_train_raw.skew():.4f}')
print(f'Kurtosis : {y_train_raw.kurtosis():.4f}')

In [ ]:
# ============================================================
# CELL 10: Visualisasi distribusi umur (histogram + boxplot)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(y_train_raw, bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribusi Age (years) — Histogram', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Umur (tahun)')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(y_train_raw.mean(), color='red', linestyle='--', label=f'Mean = {y_train_raw.mean():.1f}')
axes[0].axvline(y_train_raw.median(), color='green', linestyle='--', label=f'Median = {y_train_raw.median():.1f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(y_train_raw, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7))
axes[1].set_title('Boxplot Age (years)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Umur (tahun)')

plt.tight_layout()
plt.savefig('distribusi_umur.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisasi distribusi umur tersimpan ✅')

# ============================================================
# CELL 14: Simpan hasil preprocessing + binning
# ============================================================
import json

OUTPUT_DIR = 'preprocessed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Simpan sebagai CSV
X_train.to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(OUTPUT_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False)

# Simpan juga y_train_raw (umur asli) sebagai referensi
y_train_raw.to_csv(os.path.join(OUTPUT_DIR, 'y_train_raw.csv'), index=False)

# Simpan label mapping sebagai JSON
# Agar notebook skenario (B.1–B.6) bisa langsung menggunakan label deskriptif
# pada confusion matrix dan classification report, bukan hanya angka 0,1,2,3
with open(os.path.join(OUTPUT_DIR, 'label_names.json'), 'w') as f:
    json.dump(label_names, f, ensure_ascii=False, indent=2)

print(f'Data tersimpan di folder: {OUTPUT_DIR}/')
print(f'  - X_train.csv      : {X_train.shape}')
print(f'  - X_test.csv       : {X_test.shape}')
print(f'  - y_train.csv      : {y_train.shape} (Age_Group — target klasifikasi)')
print(f'  - y_train_raw.csv  : {y_train_raw.shape} (Age asli — referensi)')
print(f'  - label_names.json : mapping kode kelas → label deskriptif')
print('\nSemua data siap untuk digunakan di notebook skenario berikutnya! 🎯')

In [ ]:
# ============================================================
# CELL 15: Ringkasan akhir
# ============================================================
print('=' * 60)
print('RINGKASAN PREPROCESSING & BINNING')
print('=' * 60)
print(f'✅ Dataset          : Biological Age Predictive (Kaggle)')
print(f'✅ Jumlah fitur     : {X_train.shape[1]}')
print(f'✅ Sampel training  : {X_train.shape[0]}')
print(f'✅ Sampel test      : {X_test.shape[0]}')
print(f'✅ Target asli      : Age (years) → range {y_train_raw.min()}–{y_train_raw.max()}')
print(f'✅ Target baru      : Age_Group → {y_train.nunique()} kelas')
print(f'✅ Strategi binning : Domain kesehatan + kuartil data')
print()
print('Distribusi kelas akhir:')
for idx in sorted(label_names.keys()):
    count = (y_train == idx).sum()
    pct = count / len(y_train) * 100
    print(f'  Kelas {idx} — {label_names[idx]}: {count} ({pct:.1f}%)')
print()
print('Cara load di notebook lain:')
print('  X_train = pd.read_csv("preprocessed_data/X_train.csv")')
print('  X_test  = pd.read_csv("preprocessed_data/X_test.csv")')
print('  y_train = pd.read_csv("preprocessed_data/y_train.csv").squeeze()')
print()
print('  import json')
print('  with open("preprocessed_data/label_names.json") as f:')
print('      label_names = {int(k): v for k, v in json.load(f).items()}')

In [ ]:
# ============================================================
# CELL 12: Distribusi kelas akhir (value_counts)
# ============================================================
print('=== Distribusi Kelas Age_Group ===')
dist = y_train.value_counts().sort_index()
for idx, count in dist.items():
    pct = count / len(y_train) * 100
    print(f'  Kelas {idx} ({label_names[idx]}): {count} sampel ({pct:.1f}%)')

print(f'\nTotal sampel: {len(y_train)}')
print(f'Rasio terbesar/terkecil: {dist.max()/dist.min():.2f}x')

In [ ]:
# ============================================================
# CELL 13: Visualisasi distribusi kelas akhir
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
dist_sorted = y_train.value_counts().sort_index()
bar_labels = [label_names[i] for i in dist_sorted.index]

bars = axes[0].bar(bar_labels, dist_sorted.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Kelas Age_Group', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel')
axes[0].tick_params(axis='x', rotation=15)

# Tambahkan label angka di atas bar
for bar, val in zip(bars, dist_sorted.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(dist_sorted.values, labels=bar_labels, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Proporsi Kelas Age_Group', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('distribusi_age_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisasi distribusi kelas tersimpan ✅')

---
## Bagian D — Export Data untuk Notebook Skenario Berikutnya
Simpan `X_train`, `X_test`, dan `y_train` (Age_Group) agar bisa langsung di-load.

In [ ]:
# ============================================================
# CELL 14: Simpan hasil preprocessing + binning
# ============================================================
OUTPUT_DIR = 'preprocessed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Simpan sebagai CSV
X_train.to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(OUTPUT_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False)

# Simpan juga y_train_raw (umur asli) sebagai referensi
y_train_raw.to_csv(os.path.join(OUTPUT_DIR, 'y_train_raw.csv'), index=False)

print(f'Data tersimpan di folder: {OUTPUT_DIR}/')
print(f'  - X_train.csv     : {X_train.shape}')
print(f'  - X_test.csv      : {X_test.shape}')
print(f'  - y_train.csv     : {y_train.shape} (Age_Group — target klasifikasi)')
print(f'  - y_train_raw.csv : {y_train_raw.shape} (Age asli — referensi)')
print('\nSemua data siap untuk digunakan di notebook skenario berikutnya! 🎯')

In [ ]:
# ============================================================
# CELL 15: Ringkasan akhir
# ============================================================
print('=' * 60)
print('RINGKASAN PREPROCESSING & BINNING')
print('=' * 60)
print(f'✅ Dataset          : Biological Age Predictive (Kaggle)')
print(f'✅ Jumlah fitur     : {X_train.shape[1]}')
print(f'✅ Sampel training  : {X_train.shape[0]}')
print(f'✅ Sampel test      : {X_test.shape[0]}')
print(f'✅ Target asli      : Age (years) → range {y_train_raw.min()}–{y_train_raw.max()}')
print(f'✅ Target baru      : Age_Group → {y_train.nunique()} kelas')
print(f'✅ Strategi binning : Domain kesehatan + kuartil data')
print()
print('Distribusi kelas akhir:')
for idx in sorted(label_names.keys()):
    count = (y_train == idx).sum()
    pct = count / len(y_train) * 100
    print(f'  Kelas {idx} — {label_names[idx]}: {count} ({pct:.1f}%)')
print()
print('Cara load di notebook lain:')
print('  X_train = pd.read_csv("preprocessed_data/X_train.csv")')
print('  X_test  = pd.read_csv("preprocessed_data/X_test.csv")')
print('  y_train = pd.read_csv("preprocessed_data/y_train.csv").squeeze()')